# MAMoE-LoRA vs MoELoRA Adapter：路由与专家行为对比分析

本Notebook针对你提供的对比方案，重点分析：
1. 专家路由热力图（token-level聚合）
2. 专家激活频率分布
3. 专家“功能相似度”矩阵（基于路由向量而非参数直取）
4. Layer × Expert 激活分布
5. Gating logits/概率分布箱线图（以mean activation近似）
6. 路由稳定性（entropy）

并支持**双模型对比**：
- MAMoE-LoRA
- MoELoRA Adapter


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
sns.set_theme(style='whitegrid', context='talk')


In [ ]:
MAMOE_LOG = Path('analysis_outputs/mamoe/expert_routing_logs.jsonl')
MOELORA_LOG = Path('analysis_outputs/moelora/expert_routing_logs.jsonl')
OUT = Path('analysis_outputs/routing_compare')
OUT.mkdir(parents=True, exist_ok=True)

STEP_MIN = None
STEP_MAX = None

print('MAMOE exists:', MAMOE_LOG.exists(), MAMOE_LOG)
print('MOELORA exists:', MOELORA_LOG.exists(), MOELORA_LOG)


In [ ]:
def step_ok(s):
    try: s=int(s)
    except: return False
    if STEP_MIN is not None and s < STEP_MIN: return False
    if STEP_MAX is not None and s > STEP_MAX: return False
    return True

def load_jsonl(path, model_name):
    rows=[]
    if not path.exists():
        return pd.DataFrame()
    with path.open('r',encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if not line: continue
            try: o=json.loads(line)
            except: continue
            if 'step' in o and not step_ok(o['step']):
                continue
            o['model_name']=model_name
            rows.append(o)
    return pd.DataFrame(rows)

def load_vec(path_str):
    p=Path(path_str)
    if not p.exists():
        return None
    try:
        return np.load(p).reshape(-1)
    except Exception:
        return None


## 1) 加载路由日志并构建统一表

In [ ]:
df_m = load_jsonl(MAMOE_LOG, 'MAMoE-LoRA')
df_o = load_jsonl(MOELORA_LOG, 'MoELoRA-Adapter')
rt = pd.concat([df_m, df_o], ignore_index=True)

if len(rt)==0:
    raise ValueError('No routing logs found. 请先开启 enable_expert_routing_logging 并运行训练。')

for c in ['step','layer_depth','num_tokens','num_experts','routing_entropy','routing_mean_max','routing_mean_min']:
    if c in rt.columns:
        rt[c]=pd.to_numeric(rt[c], errors='coerce')

print('rows:',len(rt))
print(rt.groupby('model_name').size())
rt.head(3)


## 2) 专家路由热力图（Layer × Expert）

In [ ]:
rows=[]
for _,r in rt.iterrows():
    v=load_vec(r.get('routing_mean_path'))
    if v is None:
        continue
    for eid,val in enumerate(v):
        rows.append({
            'model_name': r['model_name'],
            'step': r['step'],
            'layer_depth': r['layer_depth'],
            'modality': r.get('modality','unknown'),
            'expert_id': eid,
            'activation': float(val),
        })
act = pd.DataFrame(rows)

if len(act)>0:
    for (m,mod),g in act.groupby(['model_name','modality']):
        piv = g.groupby(['layer_depth','expert_id'], as_index=False)['activation'].mean().pivot(index='layer_depth', columns='expert_id', values='activation').sort_index()
        plt.figure(figsize=(10,6))
        sns.heatmap(piv, cmap='viridis')
        plt.title(f'Layer x Expert Activation Heatmap | {m} | {mod}')
        plt.tight_layout(); plt.savefig(OUT/f'fig_heatmap_layer_expert_{m}_{mod}.png', dpi=180); plt.show()


## 3) 专家激活频率分布（Top-1）

In [ ]:
rows=[]
for _,r in rt.iterrows():
    h=load_vec(r.get('routing_top1_hist_path'))
    if h is None:
        continue
    for eid,val in enumerate(h):
        rows.append({
            'model_name': r['model_name'],
            'modality': r.get('modality','unknown'),
            'expert_id': eid,
            'top1_freq': float(val),
        })
hdf=pd.DataFrame(rows)

if len(hdf)>0:
    g = hdf.groupby(['model_name','modality','expert_id'], as_index=False)['top1_freq'].mean()
    g.to_csv(OUT/'table_expert_top1_freq.csv', index=False, encoding='utf-8-sig')

    plt.figure(figsize=(12,5))
    sns.barplot(data=g, x='expert_id', y='top1_freq', hue='model_name')
    plt.title('Expert top-1 activation frequency (mean)')
    plt.tight_layout(); plt.savefig(OUT/'fig_expert_top1_freq_compare.png', dpi=180); plt.show()


## 4) 专家“功能相似度”矩阵（基于路由激活轮廓）

In [ ]:
if len(act)>0:
    for m,gm in act.groupby('model_name'):
        feat = gm.pivot_table(index='expert_id', columns=['layer_depth','modality'], values='activation', aggfunc='mean').fillna(0.0)
        if feat.shape[0] >= 2:
            sim = cosine_similarity(feat.values)
            sim_df = pd.DataFrame(sim, index=feat.index, columns=feat.index)
            sim_df.to_csv(OUT/f'table_expert_similarity_{m}.csv', encoding='utf-8-sig')

            plt.figure(figsize=(7,6))
            sns.heatmap(sim_df, cmap='coolwarm', vmin=-1, vmax=1)
            plt.title(f'Expert similarity matrix | {m}')
            plt.tight_layout(); plt.savefig(OUT/f'fig_expert_similarity_{m}.png', dpi=180); plt.show()


## 5) Gating分布箱线图（按专家）

In [ ]:
if len(act)>0:
    max_expert_to_plot = min(16, int(act['expert_id'].max())+1)
    sub = act[act['expert_id'] < max_expert_to_plot].copy()

    plt.figure(figsize=(13,5))
    sns.boxplot(data=sub, x='expert_id', y='activation', hue='model_name')
    plt.title('Gating activation distribution by expert')
    plt.tight_layout(); plt.savefig(OUT/'fig_gating_boxplot_by_expert.png', dpi=180); plt.show()


## 6) 路由稳定性：Entropy 对比

In [ ]:
ent = rt.groupby(['model_name','step','modality'], as_index=False)['routing_entropy'].mean()
ent.to_csv(OUT/'table_entropy_step_modality.csv', index=False, encoding='utf-8-sig')

plt.figure(figsize=(12,5))
sns.lineplot(data=ent, x='step', y='routing_entropy', hue='model_name', style='modality')
plt.title('Routing entropy over training steps')
plt.tight_layout(); plt.savefig(OUT/'fig_entropy_over_steps.png', dpi=180); plt.show()

ent_g = rt.groupby(['model_name','modality'], as_index=False)['routing_entropy'].mean()
plt.figure(figsize=(8,4))
sns.barplot(data=ent_g, x='modality', y='routing_entropy', hue='model_name')
plt.title('Mean routing entropy by modality')
plt.tight_layout(); plt.savefig(OUT/'fig_entropy_global_compare.png', dpi=180); plt.show()


## 7) 可直接用于论文的图表建议

1. Layer×Expert 热力图（MAMoE vs MoELoRA，text与image分别画）。
2. Expert top-1 频率柱状图（专家使用均衡性与偏好）。
3. Expert similarity 矩阵（同质化程度）。
4. Gating分布箱线图（路由判别性）。
5. Entropy 曲线与全局柱状图（稳定性 vs 随机性）。

这些图可直接支撑“分模态专家池 + 共享专家”相比“单池MoELoRA”在路由可解释性上的优势论证。
